# Can we predict next-month arrears?
Target: `arrears_next_month`. Data: monthly billing rows per account (`data.csv`), dictionary in `columns.csv`.

Questions answered below:
1. Which columns are legitimately available at prediction time? (`final_notice_sent` is suspicious.)
2. Does the base rate drift over time, and how should we validate given that?
3. How well do simple and flexible models do on a held-out *future* period?

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from sklearn.inspection import permutation_importance
pd.set_option('display.width',140)
df=pd.read_csv('data.csv',parse_dates=['billing_month']).sort_values(['meter_ref','billing_month']).reset_index(drop=True)
print(pd.read_csv('columns.csv').to_string(index=False))
print(df.shape, 'accounts:',df.meter_ref.nunique(), 'missing:',int(df.isna().sum().sum()), 'dup account-months:',int(df.duplicated(['meter_ref','billing_month']).sum()))
df.describe().T

## 1. Target, drift and a leakage check

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(12,3.5))
m=df.groupby('billing_month').arrears_next_month.mean(); m.plot(ax=ax[0],marker='o'); ax[0].set_title('Arrears rate by billing month')
df.groupby('billing_month').size().plot(ax=ax[1]); ax[1].set_title('Rows (accounts) per month')
plt.tight_layout(); plt.show()
print('Overall rate: %.3f'%df.arrears_next_month.mean())
print(df.groupby('final_notice_sent').arrears_next_month.agg(['mean','size']))
print(df.groupby('late_payments_12m').arrears_next_month.agg(['mean','size']))
print(df.groupby('tariff').arrears_next_month.agg(['mean','size']))

**Findings**
- **`final_notice_sent` is leakage.** Per the dictionary it is set "in the billing cycle that followed" — i.e. *after* the billing month, in the same window in which arrears is determined. It is essentially a consequence of the outcome (86.5% arrears when 1 vs 1.4% when 0). It would not be known at prediction time, so it is **excluded** from the real model. I show it below only to quantify how misleading it would be.
- **Strong non-stationarity.** The arrears rate is ~2–4% through 2024 and climbs to ~25% by late 2025. Random k-fold CV would mix periods and give optimistic, misleading results; I use a **temporal split** (train on the past, test on the most recent months).
- The panel grows from 36 to 1375 accounts, so older months have few rows.

## 2. Features (only information available at the end of the billing month)

In [ ]:
d=df.copy()
d['price_per_kwh']=d.bill_eur/d.kwh
d['kwh_per_occ']=d.kwh/d.occupants
d['month']=d.billing_month.dt.month
g=d.groupby('meter_ref')
# strictly-past history per account; NaN where no history
d['bill_prev']=g.bill_eur.shift(1)
d['late_prev']=g.late_payments_12m.shift(1)
d['bill_chg']=d.bill_eur/d.bill_prev-1
d['late_chg']=d.late_payments_12m-d.late_prev
d['kwh_vs_acct_mean']=d.kwh/g.kwh.transform(lambda s:s.shift(1).expanding().mean())
d=pd.concat([d,pd.get_dummies(d.tariff,prefix='tariff',dtype=float)],axis=1)
BASE=['occupants','kwh','bill_eur','late_payments_12m','price_per_kwh','kwh_per_occ','month']+[x for x in d if x.startswith('tariff_')]
HIST=['bill_chg','late_chg','kwh_vs_acct_mean']
TARGET='arrears_next_month'
cut_test=pd.Timestamp('2025-10-01'); cut_val=pd.Timestamp('2025-07-01')
train=d[d.billing_month<cut_val]; val=d[(d.billing_month>=cut_val)&(d.billing_month<cut_test)]; test=d[d.billing_month>=cut_test]
for n,s in [('train',train),('val',val),('test',test)]:
    print(n,len(s),s.billing_month.min().date(),'->',s.billing_month.max().date(),'rate %.3f'%s[TARGET].mean())
print('test accounts also seen in train: %.0f%%'%(100*test.meter_ref.isin(train.meter_ref).mean()))

Split: train < Jul 2025, validation Jul–Sep 2025 (used to compare feature sets/models), test Oct–Dec 2025 (final check). The test period has a higher base rate than training — the realistic situation for a deployed model.

In [ ]:
def evaluate(model,feats,tr,te):
    model.fit(tr[feats],tr[TARGET]); p=model.predict_proba(te[feats])[:,1]; y=te[TARGET]
    k=int(len(te)*0.1); top=np.argsort(-p)[:k]
    return dict(AUC=roc_auc_score(y,p),AP=average_precision_score(y,p),Brier=brier_score_loss(y,p),
                prec_top10pct=y.values[top].mean(),mean_pred=p.mean()),p
def models():
    return {'logistic':make_pipeline(StandardScaler(),LogisticRegression(max_iter=1000)),
            'hist_gbm':HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200,random_state=0)}
def prep(mn,*frames):  # logistic needs imputation; GBM handles NaN natively
    return [f.fillna(0) if mn=='logistic' else f for f in frames]
sets={'late_payments only':['late_payments_12m'],'base':BASE,'base+history':BASE+HIST,
      'LEAKY: base+final_notice':BASE+['final_notice_sent']}
rows=[]
for sn,f in sets.items():
    for mn,m in models().items():
        tr,va=prep(mn,train,val); r,_=evaluate(m,f,tr,va); rows.append(dict(features=sn,model=mn,**r))
print('Validation (Jul-Sep 2025); outcome rate %.3f = AP of a no-skill model, AUC 0.5'%val[TARGET].mean())
pd.DataFrame(rows).round(3)

In [ ]:
# Final test: legitimate feature sets only, trained on train+val
rows=[]; preds={}
trval=pd.concat([train,val])
for sn in ['late_payments only','base','base+history']:
    for mn,m in models().items():
        tr,te=prep(mn,trval,test); r,p=evaluate(m,sets[sn],tr,te)
        rows.append(dict(features=sn,model=mn,**r)); preds[(sn,mn)]=p
print('TEST (Oct-Dec 2025); base rate %.3f (= AP of no-skill)'%test[TARGET].mean())
res=pd.DataFrame(rows).round(3); res

In [ ]:
# Calibration and importance for the best model by test AUC
best=res.sort_values('AUC').iloc[-1]; key=(best.features,best.model); print('best:',key)
p=preds[key]; y=test[TARGET].values
q=pd.qcut(p,10,duplicates='drop'); cal=pd.DataFrame({'pred':p,'obs':y}).groupby(q,observed=True).mean()
plt.figure(figsize=(4.5,4)); plt.plot(cal.pred,cal.obs,'o-'); plt.plot([0,1],[0,1],'k--'); plt.xlabel('mean predicted'); plt.ylabel('observed'); plt.title('Calibration by decile (test)'); plt.show()
f=sets[best.features]; m=models()[best.model]; tr,te=prep(best.model,trval,test)
m.fit(tr[f],tr[TARGET])
pi=permutation_importance(m,te[f],te[TARGET],scoring='roc_auc',n_repeats=5,random_state=0)
pd.Series(pi.importances_mean,f).sort_values(ascending=False).round(4).head(8)

## 3. Conclusions
Figures below come from a run of these exact cells (re-run the notebook to refresh outputs).

- **Yes, there is real but moderate signal.** On the held-out Oct–Dec 2025 period (25.1% arrears) the best legitimate models reach **AUC ≈ 0.74–0.76** and AP ≈ 0.55 (vs 0.25 for no skill). Among the 10% of accounts scored highest, ~68–70% are actually in arrears — useful for prioritising outreach.
- **Most of the signal is `late_payments_12m`** (AUC ≈ 0.70 alone; arrears rate rises from 7% at 0 late payments to ~56% at 3+). Usage, bill, tariff, occupants and account-history features add only a few AUC points; gradient boosting barely beats logistic regression.
- **`final_notice_sent` gives AUC ≈ 0.97 but must not be used**: it is recorded after the billing month, in the window where arrears itself happens, so it is a symptom of the outcome rather than a predictor.
- **Calibration is poor under drift.** The arrears rate rose from ~2–4% (2024) to ~25% (late 2025), so models trained mostly on earlier data under-predict (mean predicted 0.06–0.14 vs 0.25 observed; the GBM's Brier score is worse than predicting the base rate). Ranking is usable; the probabilities are not, without recalibration on recent data or frequent retraining.

**Caveats / next steps:** the cause of the 2025 rise is unknown (a regime change would limit any model); only 6 months of test data; 82% of test accounts also appear in training; check what 'arrears' means operationally and add features such as payment history, arrears history and income/deprivation proxies if obtainable.

**Note:** the sandbox this was built in blocked starting a Jupyter kernel, so the saved file has no stored outputs — numbers above were produced by running the same code in a plain Python process.